# Busqueda de arquitectura neuronal (NAS) empleando Optuna

Instalamos la libreria Optuna que no esta previamente instalada en Colab. https://optuna.readthedocs.io/en/stable/tutorial/index.html

In [9]:
!pip install optuna

Importamos las librerias que vasmo a utilizar

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## Red MultiCapa

Vamos a trabajar con la base de datos MNIST para realizar ejemplos de NAS con Optuna

In [11]:
# Configuración de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Carga única de datos
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

Permitimos que el modelo pueda generar de manera aleatoria el numero de capas, la cantidad de neuronas y la funcion de activación. El modelo es una red neuronal multicapa

In [12]:
def create_mlp_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = [nn.Flatten()]
    in_features = 28 * 28

    # Selección de activación (Ejemplo 2)
    activation_name = trial.suggest_categorical("activation", ["ReLU", "LeakyReLU", "Tanh"])
    activation_func = getattr(nn, activation_name)()

    for i in range(n_layers):
        # Selección de neuronas (Ejemplo 1)
        out_features = trial.suggest_int(f"n_units_l{i}", 32, 256)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(activation_func)
        in_features = out_features

    layers.append(nn.Linear(in_features, 10))
    return nn.Sequential(*layers).to(device)

Adicinalmente, creamos una clase que gestione el modelo basa en bloques de celdas, en este caso convolucionales y max pooling

In [13]:
def objective(trial):
    #Construye modelo MLP
    model = create_mlp_model(trial)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # Entrenamiento rápido (1 época para demostración)
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        if batch_idx > 100: break # Limitar para rapidez
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    # Evaluación
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = correct / len(test_loader.dataset)
    return accuracy

In [14]:
## Ejecución
study_mlp = optuna.create_study(direction="maximize")
study_mlp.optimize(lambda t: objective(t), n_trials=20)

print(f"Mejor Accuracy MLP: {study_mlp.best_value:.4f}")

[I 2026-05-12 23:43:20,606] A new study created in memory with name: no-name-4041165d-e95a-4ee3-aa0d-a1608379acd9
[I 2026-05-12 23:43:25,815] Trial 0 finished with value: 0.9146 and parameters: {'n_layers': 2, 'activation': 'LeakyReLU', 'n_units_l0': 186, 'n_units_l1': 148}. Best is trial 0 with value: 0.9146.
[I 2026-05-12 23:43:30,877] Trial 1 finished with value: 0.912 and parameters: {'n_layers': 2, 'activation': 'ReLU', 'n_units_l0': 175, 'n_units_l1': 176}. Best is trial 0 with value: 0.9146.
[I 2026-05-12 23:43:35,707] Trial 2 finished with value: 0.9151 and parameters: {'n_layers': 2, 'activation': 'ReLU', 'n_units_l0': 253, 'n_units_l1': 244}. Best is trial 2 with value: 0.9151.
[I 2026-05-12 23:43:41,149] Trial 3 finished with value: 0.9165 and parameters: {'n_layers': 3, 'activation': 'LeakyReLU', 'n_units_l0': 240, 'n_units_l1': 133, 'n_units_l2': 146}. Best is trial 3 with value: 0.9165.
[I 2026-05-12 23:43:45,988] Trial 4 finished with value: 0.9211 and parameters: {'n_la

Mejor Accuracy MLP: 0.9211


## Red basada en bloques de celda

Para esta caso se emplean dos clases, una que construye la celda (NASCell) con sus operaciones internas, y otra clase donde se construye la red a partir de diferentes celdas (SearchableNet).

In [17]:
# Componentes de la Celda
class NASCell(nn.Module):
    def __init__(self, in_channels, out_channels, op_type, use_bn, activation):
        super(NASCell, self).__init__()
        ops = []

        # Selección de Operación
        if op_type == "conv3x3":
            ops.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        elif op_type == "conv5x5":
            ops.append(nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2))
        elif op_type == "maxpool":
            ops.append(nn.MaxPool2d(kernel_size=3, stride=1, padding=1))
            if in_channels != out_channels:
                ops.append(nn.Conv2d(in_channels, out_channels, kernel_size=1))

        # Batch Normalization Opcional
        if use_bn:
            ops.append(nn.BatchNorm2d(out_channels))

        # Activación Configurable
        ops.append(nn.ReLU() if activation == "relu" else nn.SiLU())

        self.cell = nn.Sequential(*ops)

    def forward(self, x):
        return self.cell(x)

#Modelo Definido por Optuna
class SearchableNet(nn.Module):
    def __init__(self, trial, num_cells=2):
        super(SearchableNet, self).__init__()
        self.layers = nn.ModuleList()
        current_channels = 1 # MNIST es monocromático

        for i in range(num_cells):
            out_channels = trial.suggest_int(f"c{i}_out_channels", 8, 32)
            op_type = trial.suggest_categorical(f"c{i}_op", ["conv3x3", "conv5x5", "maxpool"])
            use_bn = trial.suggest_categorical(f"c{i}_bn", [True, False])
            activation = trial.suggest_categorical(f"c{i}_act", ["relu", "silu"])

            self.layers.append(NASCell(current_channels, out_channels, op_type, use_bn, activation))
            current_channels = out_channels

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(current_channels, 10)
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.classifier(x)

#Función Objetivo de Optuna
def objective(trial):
    # Hiperparámetros de entrenamiento
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    # Carga de datos rápida
    train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform),
                              batch_size=batch_size, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SearchableNet(trial, 4).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Entrenamiento simplificado (1 Época para velocidad de búsqueda)
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        if batch_idx > 100: break # Limitar para el ejemplo
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    # Evaluación (Precisión)
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(train_loader):
            if batch_idx > 20: break
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    accuracy = correct / (20 * batch_size)
    return accuracy

In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("\nMejor arquitectura encontrada:")
print(study.best_params)

[I 2026-05-12 23:52:07,412] A new study created in memory with name: no-name-1ff6300d-f75a-4e68-b497-543766191935
[I 2026-05-12 23:52:10,982] Trial 0 finished with value: 0.407421875 and parameters: {'lr': 0.0020597836939341774, 'batch_size': 128, 'c0_out_channels': 8, 'c0_op': 'conv3x3', 'c0_bn': True, 'c0_act': 'silu', 'c1_out_channels': 28, 'c1_op': 'conv3x3', 'c1_bn': False, 'c1_act': 'relu', 'c2_out_channels': 23, 'c2_op': 'maxpool', 'c2_bn': False, 'c2_act': 'silu', 'c3_out_channels': 21, 'c3_op': 'conv3x3', 'c3_bn': False, 'c3_act': 'silu'}. Best is trial 0 with value: 0.407421875.
[I 2026-05-12 23:52:13,037] Trial 1 finished with value: 0.81875 and parameters: {'lr': 0.004512579503116383, 'batch_size': 64, 'c0_out_channels': 8, 'c0_op': 'conv5x5', 'c0_bn': True, 'c0_act': 'relu', 'c1_out_channels': 30, 'c1_op': 'maxpool', 'c1_bn': True, 'c1_act': 'silu', 'c2_out_channels': 25, 'c2_op': 'conv3x3', 'c2_bn': True, 'c2_act': 'relu', 'c3_out_channels': 27, 'c3_op': 'conv5x5', 'c3_bn


Mejor arquitectura encontrada:
{'lr': 0.009680996297571222, 'batch_size': 128, 'c0_out_channels': 27, 'c0_op': 'conv5x5', 'c0_bn': False, 'c0_act': 'silu', 'c1_out_channels': 23, 'c1_op': 'conv5x5', 'c1_bn': False, 'c1_act': 'relu', 'c2_out_channels': 28, 'c2_op': 'conv3x3', 'c2_bn': True, 'c2_act': 'silu', 'c3_out_channels': 16, 'c3_op': 'maxpool', 'c3_bn': True, 'c3_act': 'relu'}
